In [ ]:
import sys
path_to_add = "/data/biophys/schimmenti/Repositories/MHN-Bif/"
if path_to_add not in sys.path:
    sys.path.insert(0, path_to_add)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from mhnlib.dynamics import Dynamics, DualDynamics
import mhnlib.utils as mhn_utils

In [ ]:
X = pd.read_csv("GSE114397/GSE114397_HMLE_TGFb.tsv", sep="\t", index_col=0).astype(float)

In [ ]:
markers = ["CDH1", "VIM", "FN1", "ZEB1"]
marker_indices = []
for marker in markers:
    if marker in X.columns:
        marker_indices.append(X.columns.get_loc(marker))
    else:
        marker_indices.append(None)

In [ ]:
meta = X.index.to_series().str.split("_", expand=True)
meta.columns = ["day", "replicate"]

meta["day"] = meta["day"].str.replace("day", "").astype(int)
meta["replicate"] = meta["replicate"].str.replace("rep", "").astype(int)

In [ ]:
library_size = X.sum(axis=1)
X_norm = X.div(library_size, axis=0) * library_size.median()

In [ ]:
X_pca_model = PCA(n_components=20).fit(X_norm)

In [ ]:
X_pca = X_pca_model.transform(X_norm)

In [ ]:
X_pca_patterns_mean, X_pca_patterns_scale, X_pca_patterns = mhn_utils.shift_and_rms(torch.tensor(X_pca, dtype=torch.float32))
biases = -0.5* X_pca_patterns.pow(2).sum(dim=1)

In [ ]:
cell_dyn = Dynamics(X_pca_patterns.to(device), biases.to(device))

In [ ]:
betas = torch.logspace(0, 1, 50)
batch_size = 1000
num_ics = X_pca_patterns.shape[0]
x_all_fps = []
for batch_start in range(0, num_ics, batch_size):
    torch.cuda.empty_cache()
    batch_end = min(batch_start + batch_size, num_ics)
    x_all_fps.append(cell_dyn.discrete_num_iters(X_pca_patterns[batch_start:batch_end].to(device),  betas.to(device), num_iters=100, verbose=True).cpu())
x_all_fps = torch.cat(x_all_fps, dim=1)

In [ ]:
for beta_idx in range(len(betas)):
    x_fps = x_all_fps[beta_idx]
    x_pca_fps = x_fps*X_pca_patterns_scale + X_pca_patterns_mean
    x_orig_fps = X_pca_model.inverse_transform(x_pca_fps.numpy())
    x_orig_fps = torch.tensor(x_orig_fps, dtype=torch.float32)
    
        
    fig = plt.figure(figsize=(4*len(markers), 4))
    for i, marker_idx in enumerate(marker_indices):
        if marker_idx is not None:
            ax = fig.add_subplot(1, len(markers), i+1, projection='3d')
            ax.scatter(x_fps[:, 0], x_fps[:, 1], x_fps[:, 2], c=x_orig_fps[:, marker_idx], label="FPs (PCA)")
            ax.set_title(f"{markers[i]} (Beta: {betas[beta_idx]:.2f})")
            ax.set_xlabel("Expression")
            ax.set_ylabel("Expression")
            ax.legend()
    plt.show()